In [ ]:
!pip install docling-core[chunking]
!pip install docling-core[chunking-openai]
!pip install docling
!pip install -qU pip docling transformers

In [1]:
def pretty(obj):
  import json
  print(json.dumps(obj.__dict__,indent=2,default=str))

In [2]:
import pandas as pd
print(pd.__version__)

2.3.3


In [66]:
import logging
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableFormerMode,
    PdfPipelineOptions
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.layout_model_specs import DOCLING_LAYOUT_HERON_101
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# Configure accelerator options for GPU


_log = logging.getLogger(__name__)

# Faster rendering
IMAGE_RESOLUTION_SCALE = 2.0

input_doc_path = r"C:\Users\hp\OneDrive\Desktop\rag_learning_materials\Vanilla_document_based_rag\Boericke_materia_medica.pdf"


def main():
    logging.basicConfig(level=logging.INFO)

    # Use threaded options (important)
    pipeline_options = PdfPipelineOptions()

    # ---------- OCR ----------
    pipeline_options.do_ocr = False  

    # ---------- Layout ----------
    pipeline_options.layout_options.model_spec = DOCLING_LAYOUT_HERON_101


    pipeline_options.do_table_structure = False
    pipeline_options.table_structure_options.do_cell_matching = False
    #pipeline_options.table_structure_options.mode = TableFormerMode.FAST

    # ---------- Images (disable for speed) ----------
    #pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = False
    pipeline_options.generate_picture_images = False
    # ---------- Disable enrichment (prevents heavy backend loading) ----------
    pipeline_options.do_formula_enrichment = False
    pipeline_options.do_code_enrichment = False
    pipeline_options.do_picture_description = False
    pipeline_options.do_picture_classification = False

    # ---------- Memory optimization ----------
    pipeline_options.generate_parsed_pages = False

    # ---------- Thread tuning (optional speed boost) ----------

    # Create converter
    doc_converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options,backend=PyPdfiumDocumentBackend)
        }
    )

    return doc_converter

In [67]:
doc_converter = main()

In [68]:
#Load all the necessary models and initialize the pipeline (important to do this before processing documents to avoid loading models during processing which can cause timeouts)
import time
start_time = time.time()
doc_converter.initialize_pipeline(InputFormat.PDF)
init_runtime = time.time() - start_time
_log.info(f"Pipeline initialized in {init_runtime:.2f} seconds.")

INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash e9aef92bf762cd32ca4759c87ae413ad
c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Lib\site-packages\docling\pipeline\standard_pdf_pipeline.py:453: DeprecationWarning: This field is deprecated. Use `generate_page_images=True` and call `TableItem.get_image()` to extract table images from page images.
  or self.pipeline_options.generate_table_images
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:__main__:Pipeline initialized in 2.52 seconds.


In [69]:
# Process document
start_time = time.time()
conv_res = doc_converter.convert(input_doc_path)

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 86974a4c892cd1020309bc47ccc25cf2
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document Boericke_materia_medica.pdf
INFO:docling.document_converter:Finished converting document Boericke_materia_medica.pdf in 2100.95 sec.


In [90]:
import json
import ast
length = len(conv_res.__dict__['document'].__dict__['texts'])
for i in range(length):
    label = json.dumps(list(conv_res.__dict__['document'].__dict__['texts'][i].__dict__['label']), indent=2,default=str)
    chars = ast.literal_eval(label)   # converts string → list
    result = "".join(chars)

    print(result," - label:", conv_res.__dict__['document'].__dict__['texts'][i].__dict__['text'])

section_header  - label: Thank you for your patronage and custom
text  - label: Celtic Boar ic Boar is the lea is the leading publishe ding publisher of exceptional of exceptional electronic electronic books related to tra related to traditional knowledge and w knowledge and wisdom of the healing isdom of the healing arts and arts and sciences.
text  - label: Our reprints reprints of classic and hard-to-f of classic and hard-to-find texts are ind texts are carefully e carefully edited and presented presented for comfortable for comfortable on-screen on-screen reading, and can be and can be printed as printed as beautiful reference texts.
text  - label: Visit celticboar.com to view our growing growing catalogue including:
text  - label: Hippocrates Hahnemann Galen Culpeper
text  - label: Paracelsus
text  - label: Physik Homeopathy
text  - label: Herbals
text  - label: Apothecary Recipes
text  - label: Alchemy
text  - label: And more…
section_header  - label: Homoeopathic Materia Medica


In [136]:
import json
import ast
print(conv_res.__dict__['document'].__dict__['texts'][1].__dict__.keys())

dict_keys(['self_ref', 'parent', 'children', 'content_layer', 'meta', 'label', 'prov', 'source', 'comments', 'orig', 'text', 'formatting', 'hyperlink', 'level'])


In [70]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from docling.chunking import HybridChunker, HierarchicalChunker

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 500  # set to a small number for illustrative purposes

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer` for HF case
)

In [ ]:
#Document stores data → Serializer decides how to present it
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    TripletTableSerializer
)
from docling_core.transforms.serializer.markdown import MarkdownParams



class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(
                image_placeholder="<!-- image -->"
            ),
        )
    
chunker = HybridChunker(
    tokenizer=tokenizer,
    serializer_provider=MDTableSerializerProvider(),
    merge_peers=True,  # optional, defaults to True
)
chunk_iter = chunker.chunk(dl_doc=conv_res.document)
chunks = list(chunk_iter)

Token indices sequence length is longer than the specified maximum sequence length for this model (571 > 512). Running this sequence through the model will result in indexing errors


In [87]:
for chunk in chunks[20:30]:
    print("---- CHUNK START ----")
    print("page_no:", chunk.meta.doc_items[0].prov[0].page_no, "text:", chunker.contextualize(chunk=chunk))
    print("---- CHUNK END ----\n\n")

---- CHUNK START ----
page_no: 11 text: Monkshood
A state of fear, anxiety; anguish of mind and body. Physical and mental restlessness, fright, is the most characteristic manifestation of Aconite. Acute, sudden, and violent invasion, with fever, call for it. Does not want to be touched. Sudden and great sinking of strength. Complaints and tension caused by exposure to dry, cold weather, draught of cold air, checked perspiration, also complaints from very hot weather, especially gastro-intestinal disturbances, etc. First remedy in inflammations, inflammatory fevers. Serous membranes and muscular tissues affected markedly. Burning in internal parts; tingling, coldness and numbness. Influenza. Tension of arteries; emotional and physical mental tension explain many symptoms. When prescribing Aconite remember Aconite causes on ly functional disturbance, no evidence that it can produce tissue change––its action is brief and shows no periodicity. Its sphere is in the beginning of an acute dis

In [72]:
from sentence_transformers import SentenceTransformer
import math
model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = model.tokenizer

data_objects = []
threshold = 200

for idx, chunk in enumerate(chunks):
    text = chunk.text.strip()

    # Step 1: Filter small chunks
    if len(text) <= threshold:
        continue

    # Step 3: Extract heading
    heading = None
    if hasattr(chunk.meta, "headings") and chunk.meta.headings:
        heading = chunk.meta.headings[0]

    # Step 4: Extract doc_items info (page_no + reference)
    page_no = None

    if chunk.meta.doc_items:
        first_item = chunk.meta.doc_items[0]

        # page number
        if first_item.prov:
            page_no = first_item.prov[0].page_no
    
    filename = None
    if chunk.meta.origin and chunk.meta.origin.filename:
        filename = chunk.meta.origin.filename

    # Step 5: Build dictionary
    properties = {
        "Heading": heading,
        "Content": text,
        "Page_No": page_no,
        "Filename": filename
    }
    vector = model.encode(chunker.contextualize(chunk=chunk)).tolist()  # Convert numpy array to list for JSON serialization
    norm = math.sqrt(sum(x*x for x in vector))
    normalized_vector = [x / norm for x in vector]
    data_object = {
        "properties": properties,
        "vector": normalized_vector
    }
    data_objects.append(data_object)

print("Total data objects:", len(data_objects))

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.34it/s]

Total data objects: 1223


In [73]:
print(data_objects[1])

{'properties': {'Heading': 'William B Boericke', 'Content': 'William Boericke 1849-1929, US\nIn preparing the ninth edition of this work, I have followed the lines laid out for all the previous editions, namely, to present in a condensed form the homœopathic Materia Medica for practical use.\nThe book contains the well known verified characteristic symptoms of all our medicines besides other less important symptoms aiding the selection of the curative remedy, All the new medicines and essentials of the published clinical experience of the school have been added. In its present compact form it contains the maximum number of reliable Materia Medica facts in the minimum space. I have tried to give a succinct resume of the symptomatology of every medicine used in Homœopathy, including also clinical suggestions of many drugs so far not yet based on provings, thus offering the opportunity to experiment with these and by future provings discover their distinctive use and so enlarging our arma

In [75]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization, StopwordsPreset, VectorDistances
from weaviate.util import generate_uuid5

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="JudgeFlowDocs",
        vector_config=Configure.Vectors.self_provided(
            name="Content",
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.DOT,
                
            )
        ),
        properties=[
        Property(
            name="Heading",
            data_type=DataType.TEXT,
            tokenization=Tokenization.TRIGRAM,
            index_searchable=True
        ),
        Property(
            name="Content",
            data_type=DataType.TEXT,
        ),
        Property(
            name="Page_No",
            data_type=DataType.INT
        ),
        Property(
            name="Filename",
            data_type=DataType.TEXT,
            tokenization=Tokenization.WORD,
            index_filterable=True
        )
    ],
    inverted_index_config=Configure.inverted_index(
        bm25_b=0.7,
        bm25_k1=1.25,
        index_null_state=True,
        index_property_length=True,
        index_timestamps=True,
        stopwords_preset=StopwordsPreset.EN,
        stopwords_additions=["1","2","3","4","5","6","7","8","9","0"],
    ),
    )

    judge_flow_collection = client.collections.use("JudgeFlowDocs")
    with judge_flow_collection.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj['properties'], vector=obj['vector'], uuid=generate_uuid5(obj['properties']))
            if batch.number_errors > 10:
                print("Batch import stopped due to excessive errors.")
                break 
    failed_objects = judge_flow_collection.batch.failed_objects
    if failed_objects:
        print(f"Number of failed imports: {len(failed_objects)}")
        print(f"First failed object: {failed_objects[0]}")
    print(f"Imported & vectorized {len(judge_flow_collection)} objects into the JudgeFlowDocs collection")

INFO:httpx:HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"
c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\hp\AppData\Local\Temp\ipykernel_7604\2703070845.py:50: ResourceWarning: unclosed <socket.socket fd=4084, family=23, type=1, proto=0, laddr=('::1', 53292, 0, 0), raddr=('::1', 8080, 0, 0)>
  judge_flow_collection = client.collections.use("JudgeFlowDocs")
INFO:httpx:HTTP Request: GET http://localhost:8080/v1/schema/JudgeFlowDocs "HTTP/1.1 200 

Imported & vectorized 1223 objects into the JudgeFlowDocs collection


In [74]:
client.collections.delete("JudgeFlowDocs")


INFO:httpx:HTTP Request: DELETE http://localhost:8080/v1/schema/JudgeFlowDocs "HTTP/1.1 200 OK"


In [77]:
client = weaviate.connect_to_local()

INFO:httpx:HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"


In [16]:
from weaviate.classes.query import BM25Operator, MetadataQuery, GroupBy
judge_flow_collection = client.collections.use("JudgeFlowDocs")
# group_by = GroupBy(
#     prop="Heading",
#     objects_per_group=3,
#     number_of_groups=2,
# )
response = judge_flow_collection.query.bm25(
    query="intro",
    operator=BM25Operator.or_(minimum_match=1),
    query_properties=["Heading"],
    return_metadata=MetadataQuery(score=True),
    auto_limit=2
    #group_by=group_by
)
for o in response.objects:
    print(o.properties)
    print(o.metadata.score)

{'content': 'To address these challenges, we introduce JUDGEFLOW, an Evaluation-Judge-Optimization-Update pipeline. First, we incorporate reusable and configurable logic blocks into agentic workflows. These blocks capture three fundamental forms of logic: sequential, loop, and conditional, which are able to broadly represent code-based workflows. Compared with operators, which abstract specific agentic operations or functionalities (Zhang et al. , 2025b), logic blocks serve as higher-level structural abstractions. By introducing logic blocks that abstract such common control structures, we retain the structural diversity of code-represented workflows while providing an intermediate level of abstraction between operators and workflows. This additional layer facilitates interpretability and exposes more meaningful diagnostic information for subsequent optimization.\nSecond, we incorporate a dedicated Judge module that analyzes the execution trace, with particular emphasis on failed runs.

In [ ]:
judge_flow_collection = client.collections.use("JudgeFlowDocs")
response = judge_flow_collection.query.near_vector(
    near_vector=model.encode("Give me the algorithm of evaluation judge").tolist(),
    auto_limit=2,
    return_metadata=MetadataQuery(distance=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.65it/s]

{'content': 'Building on the representation of workflow using logic blocks, JUDGEFLOW incorporates a dedicated Judge module and implements an iterative Evaluation-Judge-Optimization-Update pipeline for the efficient optimization of agentic workflows and continues until a predefined maximum number of optimization rounds are met.', 'filename': 'JudgeFlow.pdf', 'heading': '3.2 JUDGEFLOW', 'page_No': 5}
0.37912678718566895
{'content': "The combined Evaluation-Judge stage, detailed in Algorithm 1, processes each input query from the dataset. If the workflow W fails on a given query, the stage identifies and logs specific problematic block within W. This provides targeted diagnostic signals for subsequent workflow optimization, enabling a more efficient and focused approach on refining these identified weak logic to improve overall optimization efficiency.\nSpecifically, for each input query q (with a corresponding ground-truth answer a), we have {a ′ i } M i=1 = ϕ exe (q, W), and score s = 

In [80]:
from weaviate.classes.query import HybridFusion
judge_flow_collection = client.collections.use("JudgeFlowDocs")
vector=model.encode("""A remedy is described as acting primarily on the sexual organs, producing a state of sexual neurasthenia, with symptoms including profuse nightly emissions, lascivious dreams, and urinary burning followed by dribbling.

However, within the same context, another substance is discussed as being widely used in large-scale public health campaigns, noted for being palatable, requiring no prior preparation, and capable of removing up to 95% of a specific parasitic infestation in a single dose.

 Identify:

The remedy responsible for the sexual and urinary symptoms
The substance used in mass treatment campaigns
The disease targeted by that mass treatment""").tolist()
norm = math.sqrt(sum(x*x for x in vector))
normalized_vector = [x / norm for x in vector]

response = judge_flow_collection.query.hybrid(
    query="""A remedy is described as acting primarily on the sexual organs, producing a state of sexual neurasthenia, with symptoms including profuse nightly emissions, lascivious dreams, and urinary burning followed by dribbling.

However, within the same context, another substance is discussed as being widely used in large-scale public health campaigns, noted for being palatable, requiring no prior preparation, and capable of removing up to 95% of a specific parasitic infestation in a single dose.

Identify:

The remedy responsible for the sexual and urinary symptoms
The substance used in mass treatment campaigns
The disease targeted by that mass treatment""", 
    auto_limit=2,
    bm25_operator=BM25Operator.or_(minimum_match=1),
    query_properties=["Heading"],
    fusion_type=HybridFusion.RELATIVE_SCORE,
    alpha=1,  # optional, defaults to 0.5 (equal weighting)
    vector=normalized_vector,
    target_vector="Content"
    )

for o in response.objects:
    print(o.properties)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.17it/s]

{'content': 'tract. Loosens secretion, relieves tightened feeling, makes expectoration easy). Neurotic coughs. Huskiness of public speakers, and singers. Cystitis when urine is alkaline and offensive.\nOnonis spinosa-Rest Harrow––(Diuretic, Lithontriptic. Chronic nephritis; diuretic effects like Juniper; calculus nosebleed, worse washing face).\nAntidote: Phos.\nDose. – –First to sixth potency.', 'filename': 'Boericke_materia_medica.pdf', 'heading': 'Homoeopathic Materia Medica', 'page_No': 534}
{'content': 'waking in morning.\nSleep.––Sleep during day; restless at night on account of heat and shock: anxious dreams.\nModalities.––Worse, immediately after eating, lying on right side: from sea bathing. Better, from pressure, motion: open air, except headache.\nRelationship.––Antidotes: Camph; Cham.\nCompare: Nat m; Puls; Sep; Amm m; Nasturtium equaticum -Water-cress–– (useful in scorbutic affections and constipation, related to strictures of urinary apparatus; supposed to be aphrodisiaca